# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library, referencing all dataset elements by their Croissant `@id` fields.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata as a Python object (not as a dictionary)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print("Dataset DOI:", metadata.identifier)
print("Keywords:", metadata.keywords)
print("License:", metadata.license)
print("Date published:", metadata.datePublished)
print("Spatial coverage:", metadata.spatialCoverage)
print("Temporal coverage:", metadata.temporalCoverage)


## 2. Data Overview
Review the dataset's available record sets (tables), fields, and their `@id` values. All references use the `@id` to uniquely identify data elements.

In [ ]:
# List available record sets and their fields (all by `@id`)
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    print("Record Sets (by @id):")
    for rs in metadata.recordSet:
        print(f"  - @id: {rs['@id']}")
        print(f"    Name: {rs.get('name', '(no name)')}")
        # List fields of each record set
        if 'field' in rs:
            fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
            print(f"    Fields:")
            for field in fields:
                if isinstance(field, dict):
                    print(f"      - @id: {field['@id']}, name: {field.get('name', '')}")
                else:
                    print(f"      - @id: {field}")
        print()
else:
    print("No record sets were found in the dataset metadata.")

## 3. Data Extraction
Load records from specific record sets. All record sets and fields are referenced via their `@id` fields, as shown in the overview above.

We'll demonstrate loading all available record sets, if any, and inspecting their first records.

In [ ]:
# Collect all record sets' @id
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for rs in metadata.recordSet:
        if isinstance(rs, dict) and '@id' in rs:
            record_sets.append(rs['@id'])

# Load all available record sets into DataFrames
dataframes = {}
for rs_id in record_sets:
    print(f"Loading records for record set @id: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        print(f"  Columns: {list(df.columns)}; Rows: {df.shape[0]}")
        dataframes[rs_id] = df
        if not df.empty:
            display(df.head())
    except Exception as e:
        print(f"  Could not load records for {rs_id}: {e}\n")

if not dataframes:
    print("No record sets with records found in this dataset.")

## 4. Exploratory Data Analysis (EDA)
In this section, we demonstrate typical data analysis steps: filtering by numeric field, normalization, and grouping by key fields—all by referencing the dataset's field `@id`s.

If there are no record sets with records, this section is informational only.

In [ ]:
# Example EDA steps (run only if we have at least one DataFrame loaded)
if dataframes:
    # Choose the first non-empty DataFrame as example
    for rs_id, df in dataframes.items():
        if not df.empty:
            first_rs_id = rs_id
            break
    print(f"Using record set @id: {first_rs_id} for EDA.")

    # Identify numeric fields by checking dtypes
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_fields:
        print("No numeric fields found for EDA.")
    else:
        # Use the first numeric field as an example
        numeric_field = numeric_fields[0]
        print(f"Analyzing numeric field (by @id): {numeric_field}")

        # Example: Filter records, e.g., values above the mean
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        norm_field = f"{numeric_field}_normalized"
        filtered_df[norm_field] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_field]].head())

        # Group by a categorical field if present
        cat_fields = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) and col != numeric_field]
        if cat_fields:
            group_field = cat_fields[0]
            print(f"Grouping by field (by @id): {group_field}")
            grouped = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            display(grouped.head())
        else:
            print("No categorical field found for grouping.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset, using only Croissant `@id` field references.

In [ ]:
import matplotlib.pyplot as plt

if dataframes:
    # Use the same example DataFrame and fields from EDA
    df = dataframes[first_rs_id]
    if numeric_fields:
        plt.figure(figsize=(6,4))
        df[numeric_field].hist(bins=15, edgecolor='black')
        plt.title(f"Distribution of {numeric_field} (by @id)")
        plt.xlabel(numeric_field)
        plt.ylabel('Frequency')
        plt.show()
        
        # If grouping field exists, show boxplot
        if cat_fields:
            import seaborn as sns
            plt.figure(figsize=(8,4))
            sns.boxplot(x=df[cat_fields[0]], y=df[numeric_field])
            plt.title(f"{numeric_field} by {cat_fields[0]}")
            plt.xlabel(cat_fields[0])
            plt.ylabel(numeric_field)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
This notebook demonstrated how to load and explore a dataset defined by a Croissant schema using the `mlcroissant` library, referencing all entities by their `@id`. Review metadata, extract and process records using the record set and field `@id`, and perform sample visualizations. You can extend this analysis by leveraging additional field information from the dataset's Croissant schema.